In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json

In [2]:
file = 'flowUpdatesStreamCMI.json'
data = json.load(open(file))

## Map to use clear names instead of MACs

In [3]:
host_map = {
 '0A-AA-00-00-00-24': 'lidar_fl',
 '0A-AA-00-00-00-2C': 'connectivity_gw',
 '0A-AA-00-00-00-20': 'zc_fl',
 '0A-AA-00-00-00-2A': 'infotainment',
 '0A-AA-00-00-00-28': 'camera_front',
 '0A-AA-00-00-00-25': 'lidar_fr',
 '0A-AA-00-00-00-21': 'zc_fr',
 '0A-AA-00-00-00-29': 'camera_rear',
 '0A-AA-00-00-00-26': 'lidar_rl',
 '0A-AA-00-00-00-22': 'zc_rl',
 '0A-AA-00-00-00-2B': 'adas',
 '0A-AA-00-00-00-23': 'zc_rr',
 '0A-AA-00-00-00-27': 'lidar_rr',
}

switch_map = {
 '0A-AA-00-00-00-06': 'sw_fl',
 '0A-AA-00-00-00-0C': 'sw_fr',
 '0A-AA-00-00-00-11': 'sw_rl',
 '0A-AA-00-00-00-16': 'sw_rr',
 '0A-AA-00-00-00-1B': 'sw_c',
}

# Topology

## Read Topology

In [6]:
# read the topology information and organize it
topology = data['finalConfig']['topology']

switches = []
device_links = dict()
hosts = dict()
host_map_ips = dict()
for link in topology:
    if link['switch'] not in switch_map:
        print("ERROR: " + link['switch'] + " not known in switch_map")
        continue
    dev1 = switch_map[link['switch']]
    if dev1 not in switches:
        switches.append(dev1)
    port1 = link['port']
    fromDev = dev1 + ':' + str(port1)
    if link['deviceLink'] is True:
        dev2 = link['device']['switch']
        if dev2 not in switch_map:
            print("ERROR: " + dev2 + " not known in switch_map")
            continue
        dev2 = switch_map[dev2]
        if dev2 not in switches:
            switches.append(dev2)
        port2 = link['device']['port']
        toDev = dev2 + ':' + str(port2)
        if dev1 not in device_links:
            device_links[dev1] = dict()
        device_links[dev1][port1] = toDev
        if dev2 not in device_links:
            device_links[dev2] = dict()
        device_links[dev2][port2] = fromDev
    else: # is host link
        hostmac = link['host']['mac']
        hostip = link['host']['ip']
        if hostmac not in host_map:
            print("ERROR: " + hostmac + " not known in host_map")
        else:
            hosts[host_map[hostmac]] = fromDev
        if hostip not in host_map_ips:
            host_map_ips[hostip] = host_map[hostmac]
print(device_links)
print(hosts)

{'sw_fl': {0: 'sw_c:0'}, 'sw_c': {0: 'sw_fl:0', 1: 'sw_fr:0', 3: 'sw_rl:0', 2: 'sw_rr:0'}, 'sw_fr': {0: 'sw_c:1'}, 'sw_rl': {0: 'sw_c:3'}, 'sw_rr': {0: 'sw_c:2'}}
{'lidar_fl': 'sw_fl:1', 'connectivity_gw': 'sw_fl:2', 'zc_fl': 'sw_fl:3', 'infotainment': 'sw_fl:4', 'camera_front': 'sw_fr:1', 'lidar_fr': 'sw_fr:2', 'zc_fr': 'sw_fr:3', 'camera_rear': 'sw_rl:1', 'lidar_rl': 'sw_rl:2', 'zc_rl': 'sw_rl:3', 'adas': 'sw_rr:1', 'zc_rr': 'sw_rr:2', 'lidar_rr': 'sw_rr:3'}


In [7]:
# parse connections
connections = []
for origin in device_links.keys():
    for port, target in device_links[origin].items():
        dir1 = origin + ':' + target.split(':')[0]
        dir2 = target.split(':')[0] + ':' + origin
        if dir1 not in connections and dir2 not in connections:
            connections.append(dir1)
connections

['sw_fl:sw_c', 'sw_c:sw_fr', 'sw_c:sw_rl', 'sw_c:sw_rr']

## Parse and write to file

In [ ]:
topology_out = "car_topology.py"
# write preamble
with open(topology_out, 'w') as f:
    f.write("#!/usr/bin/env python3\n")
    f.write("# Copyright (C) 2024 Timo Salomon for HAW Hamburg, Germany\n")
    f.write("# This file was automatically generated from omnetpp SDN4CoRE output " + file + "\n")
    f.write("# DO NOT EDIT THIS FILE\n")
    f.write("# \n")
    f.write("\n")
    f.write("import sys\n")
    f.write("from _decimal import Decimal\n")
    f.write("from networkx import grid_graph\n")
    f.write("import random\n")
    f.write("import networkx as nx\n")
    f.write("import matplotlib.pyplot as plt\n")
    f.write("from dynamic_reservation.environment.network import Network\n")
    f.write("\n")
    f.write("def car_topology(linkrate_backbone=10000000000.0, linkrate_nodes=1000000000.0, queuesize=Decimal(sys.float_info.max), output=False, nrCBSqueues=2):\n")
    f.write("    # create network\n")
    f.write("    net = Network()\n")
    f.write("\n")
    f.write("    # create switches\n")
    f.write("    bridges = [")
    for sw in switches:
        f.write("'" + sw + "', ")
    f.write("]\n")
    f.write("\n")
    f.write("    # add switches to network\n")
    for connection in connections:
        nodeA = connection.split(':')[0]
        nodeB = connection.split(':')[1]
        f.write("    net.create_link(nodeA='" + nodeA + "', nodeB='" + nodeB + "', priorities=nrCBSqueues, queue_in=queuesize, rate_out=linkrate_backbone)\n")
    f.write("\n")
    f.write("    # add hosts\n")
    f.write("    tsn_devices = [")
    for host in hosts.keys():
        f.write("'" + host + "', ")
    f.write("]\n")
    f.write("\n")
    f.write("    # add hosts to network\n")
    for host in hosts.keys():
        if "adas" in host:
            f.write("    net.create_link(nodeA='" + host + "', nodeB='" + hosts[host].split(":")[0] + "', priorities=nrCBSqueues, queue_in=queuesize, rate_out=linkrate_backbone)\n")
        else:
            f.write("    net.create_link(nodeA='" + host + "', nodeB='" + hosts[host].split(":")[0] + "', priorities=nrCBSqueues, queue_in=queuesize, rate_out=linkrate_nodes)\n")
    f.write("\n")
    f.write("    if output:\n")
    f.write("        nx.draw_planar(net.graph, with_labels=True)\n")
    f.write("        plt.show()\n")
    f.write("\n")
    f.write("    return net.graph, bridges, tsn_devices\n")


# Flows

In [4]:
IPG = 96
PCP_OFFSET = 2
TIME_DELTA = 0.000000001
OPERATION_ADD = 'add'
OPERATION_REMOVE = 'remove'

## read flows

In [9]:
# read flows and create flow list
flow_updates = dict()
flow_ids = []
for update in data['flowUpdates']:
    if 'resourceReservation' not in update:
        print("WARNING: no resourceReservation in update at " + str(update['simTime']) + " for flow " + str(update['serviceId']))
        continue
    time = update['simTime']
    while time in flow_updates:
        time += TIME_DELTA
    flow_updates[time] = dict()
    flowId = update['serviceId']
    if flowId not in flow_ids:
        flow_ids.append(flowId)
        flow_updates[time]['isMcastUpdate'] = False
    else:
        flow_updates[time]['isMcastUpdate'] = True
    flow_updates[time]['flowId'] = flowId
    flow_updates[time]['source'] = host_map_ips[update['srcHost']]
    flow_updates[time]['sink'] = host_map_ips[update['dstHost']] # todo support multiple sinks with mcast
    if update['resourceReservation']['l2FrameSize'] != 74:
        print(str(update['simTime']) + " for flow " + str(update['serviceId']) + " frame size is " + str(update['resourceReservation']['l2FrameSize']))
    flow_updates[time]['data_size'] = update['resourceReservation']['l2FrameSize']*8 + IPG
    flow_updates[time]['interval'] = update['resourceReservation']['measurementInterval']
    if update['resourceReservation']['l2FrameSize'] > 1526:
        flow_updates[time]['max_frame_size'] = 1526*8
        print("WARNING: max frame size > 1526 at " + str(update['simTime']) + " for flow " + str(update['serviceId']) + " reduced to 1526")
    else:
        flow_updates[time]['max_frame_size'] = update['resourceReservation']['l2FrameSize']*8
    flow_updates[time]['priority'] = 7 - (update['resourceReservation']['pcp']+PCP_OFFSET)
    flow_updates[time]['numFrames'] = update['resourceReservation']['numFrames']
    flow_updates[time]['path'] = []
    for hop in update['route']:
        flow_updates[time]['path'].append(switch_map[hop['switch']])

print(flow_updates)

0.0931858 for flow 1111 frame size is 1428
0.0951494 for flow 2112 frame size is 984
0.0953326 for flow 2111 frame size is 984
0.0965804 for flow 1112 frame size is 1428
0.0968303 for flow 2113 frame size is 984
0.0973808 for flow 2114 frame size is 984


## Parse and write to file

In [ ]:
flows_out = "car_flows.py"
currentState = dict()
flowReceivers = dict()
# write preamble
with open(flows_out, 'w') as f:
    f.write("#!/usr/bin/env python3\n")
    f.write("# Copyright (C) 2024 Timo Salomon for HAW Hamburg, Germany\n")
    f.write("# This file was automatically generated from omnetpp SDN4CoRE output " + file + "\n")
    f.write("# DO NOT EDIT THIS FILE\n")
    f.write("# \n")
    f.write("from dynamic_reservation.environment.flow import Flow\n")
    f.write("\n")
    f.write("def car_flow_updates():\n")
    f.write("    operations = [] \n")
    f.write("\n")
    for time in flow_updates.keys():
        # if not "lidar" in flow_updates[time]['source'] and not "camera" in flow_updates[time]['source']:
        #     continue
        # if "lidar" in flow_updates[time]['source'] or "camera" in flow_updates[time]['source']:
        #     continue
        id = flow_updates[time]['flowId']
        if flow_updates[time]['isMcastUpdate']:
            if not id in currentState:
                print("ERROR: flow " + str(id) + " is mcast update but not in currentState")
                continue
            # remove old flow first
            f.write("    # remove old flow\n")
            f.write("    operations.append(('" + OPERATION_REMOVE + "', " + currentState[id] + "))\n")
        # add new flow
        if id not in flowReceivers:
            flowReceivers[id] = []
        flowReceivers[id].append(flow_updates[time]['sink'])
        deadline = 0.01
        # if flow_updates[time]['priority'] == 0:
        #     deadline = 0.001
        f.write("    # add new flow\n")  
        flow = "Flow(" + "flowID='" + str(id) + "'" + ", source='" + flow_updates[time]['source'] + "'" + ", sinks=" + str(flowReceivers[id]) + ", data_per_interval=" + str(flow_updates[time]['data_size']) + ", sending_interval=" + str(flow_updates[time]['interval']) + ", deadline=" + str(deadline) + ", max_frame_size=" + str(flow_updates[time]['max_frame_size']) + ", priority=" + str(flow_updates[time]['priority']) + ", frames_per_interval=" + str(flow_updates[time]['numFrames']) + ", redundancy=False)"
        currentState[id] = flow
        f.write("    operations.append(('" + OPERATION_ADD + "', " + flow + "))\n")
    f.write("    return operations\n")
    f.write("\n")
    # also write a function returning the complete flow list of the final network state cached in currentState
    f.write("def car_flows():\n")
    f.write("    flows = []\n")
    for id in currentState.keys():
        f.write("    flows.append(" + currentState[id] + ")\n")
    f.write("    return flows\n")
    f.write("\n")

# Parse GCL

## Read Shaping.ini

In [67]:
# OMNeT device names to car names
name_map = {
    'zonalControllerFrontLeft': 'zc_fl',
    'adas': 'adas',
    'switchFrontLeft': 'sw_fl',
    'switchFrontRight': 'sw_fr',
    'switchCenter': 'sw_c',
    'switchRearLeft': 'sw_rl',
    'switchRearRight': 'sw_rr',
}


In [68]:
gcl_file = "shaping.ini"
gcl_period = 0.001 # 1ms
cbsGateClosed = dict()
with open (gcl_file, 'r') as f:
    for line in f:
        if not "gateControlList" in line:
            continue
        #example line:
        # *.zonalControllerFrontLeft.eth[*].shaper.gateControlList.controlList = "C,C,C,C,C,C,o,C:0;o,o,o,o,o,o,C,o:0.000004;C,C,C,C,C,C,C,C:0.000987"
        # get the device and the port
        dev = line.split('.')[1]
        port = line.split('.')[2].split('[')[1].split(']')[0]
        # extract the gateControlList steps
        gcl_strings = line.split('"')[1].split(';')
        # calculate the time where cbs prios 4 and 5 are closed
        cbs_closed = 0
        last_state = 'o'
        closed_at = 0
        for i in range(0, gcl_strings.__len__()):
            # get key with index i
            key = gcl_strings[i].split(':')[0]
            value = float(gcl_strings[i].split(':')[1])
            states = key.split(',')
            if 'C' in states[4] and 'C' in states[5]:
                if last_state == 'o':
                    closed_at = value
                last_state = 'C'
            else:
                if last_state == 'C':
                    cbs_closed += value - closed_at
                last_state = 'o'
        if last_state == 'C': ## closed until the end of the period
            cbs_closed += gcl_period - closed_at
        mapped_dev = name_map[dev]
        if mapped_dev not in cbsGateClosed:
            cbsGateClosed[mapped_dev] = dict()
        cbsGateClosed[mapped_dev][port] = cbs_closed
cbsGateClosed

{'zc_fl': {'*': 1.699999999999999e-05},
 'adas': {'*': 1.699999999999999e-05},
 'sw_fl': {'0': 1.7000000000000003e-05, '3': 1.7000000000000007e-05},
 'sw_fr': {'3': 2.0000000000000012e-05},
 'sw_c': {'0': 1.7000000000000007e-05,
  '2': 1.7000000000000007e-05,
  '1': 2.100000000000001e-05,
  '3': 2.100000000000001e-05},
 'sw_rl': {'3': 2.0000000000000012e-05},
 'sw_rr': {'0': 1.7000000000000003e-05, '2': 2.0000000000000012e-05}}

## Parse and write to file

In [69]:
# parse the connections for the gcls depending on the car topology
print(device_links)
print(hosts)
connectionsGcl = dict()
for dev in cbsGateClosed.keys():
    fromDev = dev
    if fromDev not in connectionsGcl:
        connectionsGcl[fromDev] = dict()
    if dev not in device_links:
        if dev not in hosts:
            print("ERROR: " + dev + " not in device_links or hosts")
            continue
        else:
            if cbsGateClosed[dev].keys().__len__() > 1:
                print("ERROR: " + dev + " is host but has more than one port")
                continue
            port = list(cbsGateClosed[dev].keys())[0]
            if not "*" in port and not "0" in port: 
                print("ERROR: " + dev + " is host but port is not * or 0 port:" + port)
                continue
            toDev = hosts[dev].split(':')[0]
            connectionsGcl[fromDev][toDev] = cbsGateClosed[dev]['*']
    else:
        for port in cbsGateClosed[dev].keys():
            if "*" in port:
                for knownPort in device_links[dev].keys():
                    toDev = device_links[dev][knownPort].split(':')[0]
                    connectionsGcl[fromDev][toDev] = cbsGateClosed[dev][port]
                for host in hosts.keys():
                    if hosts[host] == dev + ':' + port:
                        toDev = host
                        connectionsGcl[fromDev][toDev] = cbsGateClosed[dev][port]
            elif int(port) not in device_links[dev].keys():
                # check for reverse host connections
                found = False
                for host in hosts.keys():
                    if hosts[host] == dev + ':' + port:
                        toDev = host
                        connectionsGcl[fromDev][toDev] = cbsGateClosed[dev][port]
                        found = True
                        break
                if not found:
                    print("ERROR: " + dev + " port " + port + " not in device_links and not in hosts")
            else:
                toDev = device_links[dev][int(port)].split(':')[0]
                connectionsGcl[fromDev][toDev] = cbsGateClosed[dev][port]
connectionsGcl

{'sw_fl': {0: 'sw_c:0'}, 'sw_c': {0: 'sw_fl:0', 1: 'sw_fr:0', 3: 'sw_rl:0', 2: 'sw_rr:0'}, 'sw_fr': {0: 'sw_c:1'}, 'sw_rl': {0: 'sw_c:3'}, 'sw_rr': {0: 'sw_c:2'}}
{'lidar_fl': 'sw_fl:1', 'connectivity_gw': 'sw_fl:2', 'zc_fl': 'sw_fl:3', 'infotainment': 'sw_fl:4', 'camera_front': 'sw_fr:1', 'lidar_fr': 'sw_fr:2', 'zc_fr': 'sw_fr:3', 'camera_rear': 'sw_rl:1', 'lidar_rl': 'sw_rl:2', 'zc_rl': 'sw_rl:3', 'adas': 'sw_rr:1', 'zc_rr': 'sw_rr:2', 'lidar_rr': 'sw_rr:3'}


{'zc_fl': {'sw_fl': 1.699999999999999e-05},
 'adas': {'sw_rr': 1.699999999999999e-05},
 'sw_fl': {'sw_c': 1.7000000000000003e-05, 'zc_fl': 1.7000000000000007e-05},
 'sw_fr': {'zc_fr': 2.0000000000000012e-05},
 'sw_c': {'sw_fl': 1.7000000000000007e-05,
  'sw_rr': 1.7000000000000007e-05,
  'sw_fr': 2.100000000000001e-05,
  'sw_rl': 2.100000000000001e-05},
 'sw_rl': {'zc_rl': 2.0000000000000012e-05},
 'sw_rr': {'sw_c': 1.7000000000000003e-05, 'zc_rr': 2.0000000000000012e-05}}

In [ ]:
outfile = "car_gcls.py"
# Parse and write to file
with open(outfile, 'w') as f:
    f.write("#!/usr/bin/env python3\n")
    f.write("# Copyright (C) 2024 Timo Salomon for HAW Hamburg, Germany\n")
    f.write("# This file was automatically generated from omnetpp SDN4CoRE inifile " + gcl_file + "\n")
    f.write("# DO NOT EDIT THIS FILE\n")
    f.write("# \n")
    f.write("\n")
    f.write("from _decimal import Decimal\n")
    f.write("from networkx import grid_graph\n")
    f.write("import networkx as nx\n")
    f.write("\n")
    f.write("def set_car_gcls(graph_state):\n")
    f.write("    # create gcl list\n")
    f.write("    gcls = []\n")
    f.write("\n")
    for fromDev in connectionsGcl.keys():
        for toDev in connectionsGcl[fromDev].keys():
            # example config for a connection from 1 to 3
            # nx.set_edge_attributes(graph_state['simple_graph'], {(1, 3): {'gcl': (Decimal(0.0005), Decimal(0.00005))}})
            f.write("    nx.set_edge_attributes(graph_state['simple_graph'], {('" + fromDev + "', '" + toDev + "'): {'gcl': (Decimal(" + format(gcl_period, ".3f") + "), Decimal(" + format(connectionsGcl[fromDev][toDev], ".6f") + "))}})\n")
    f.write("\n")
    f.write("    return gcls\n")
    f.write("\n")


# Validation of Framework Results

In [71]:
reservedFlows = ['4049', '4021', '6077', '4022', '3026', '4011', '6045', '6078', '4005', '6033', '6065', '5004', '3008', '4014', '3020', '7002', '6053', '4001', '4046', '5002', '3038', '3011', '3031', '7012', '3041', '4051', '6058', '4006', '4035', '4029', '4048', '6004', '3010', '4009', '4041', '3027', '4017', '4042', '3004', '4038', '6061', '4054', '6034', '6036', '4020', '7009', '7001', '6060', '6070', '6057', '6035', '1111', '3034', '4034', '6023', '6014', '6041', '3006', '7003', '6018', '5006', '6020', '3015', '3003', '5000', '6010', '3009', '6017', '4033', '4057', '3021', '4044', '6062', '6042', '6049', '4060', '7011', '4061', '6015', '3022', '7010', '6032', '6012', '6052', '3028', '4007', '6044', '6013', '4040', '6048', '6031', '2112', '6024', '6075', '6021', '6046', '3014', '2111', '7006', '4004', '3005', '7005', '4012', '4047', '6066', '3023', '3001', '3012', '3042', '6000', '6003', '4027', '3035', '3018', '3040', '7004', '3029', '4023', '6038', '4000', '6068', '6064', '6072', '6026', '4039', '4050', '6069', '6006', '4028', '6019', '3037', '3030', '4019', '6025', '4015', '3002', '6051', '4025', '4013', '7007', '3019', '6039', '6028', '6067', '6001', '3032', '4053', '6008', '6011', '6055', '6043', '4058', '6073', '4026', '4056', '4052', '6059', '4018', '4059', '6007', '6029', '6037', '4031', '5001', '3039', '4037', '6054', '3025', '6063', '6002', '6040', '4010', '6022', '4002', '6071', '7013', '3024', '6005', '3007', '3013', '6076', '4030', '7000', '3016', '4008', '3033', '6047', '7008', '3017', '6056', '3000', '5005', '4045', '4055', '4024', '4016', '6016', '4043', '4036', '6030', '6009', '6027', '5003', '6074', '4003', '6050', '3036', '4032']

In [72]:
for flow in flow_ids:
    flowstr = str(flow)
    if flowstr not in reservedFlows:
        print("WARNING: flow " + flowstr + " is in reservedFlows but not in flow_ids")

In [75]:
lines = {
"(sw_fl, sw_c) in bit/s: 92646041.96, 157282064.11",
"(sw_fl, zc_fl) in bit/s: None, 418455830.98",
"(sw_fl, infotainment) in bit/s: None, 114332074.65",
"(sw_c, sw_fl) in bit/s: None, 395381264.92",
"(sw_c, sw_fr) in bit/s: None, 379105515.46",
"(sw_c, sw_rl) in bit/s: None, 147876635.30",
"(sw_c, sw_rr) in bit/s: 1303692807.23, 328649252.56",
"(sw_fr, sw_c) in bit/s: 298316497.86, 152621162.75",
"(sw_fr, zc_fr) in bit/s: None, 390200072.63",
"(sw_rl, sw_c) in bit/s: 298316497.86, 19916520.66",
"(sw_rl, zc_rl) in bit/s: None, 153175235.34",
"(sw_rr, sw_c) in bit/s: None, 200810105.01",
"(sw_rr, adas) in bit/s: 1640390405.58, None",
"(sw_rr, zc_rr) in bit/s: None, 343064919.81",
}

In [76]:
for line in lines:
    connection = line.split(")")[0].strip().replace("(", "").split(",")
    reservation = line.split(":")[-1].strip().split(",")
    fromDev = connection[0].strip()
    toDev = connection[1].strip()
    bw = 0
    for reservedBw in reservation:
        if "None" in reservedBw:
            continue
        bw += float(reservedBw)
    print(fromDev, toDev, format(bw/1000000, ".3f"), "Mbit/s")


sw_c sw_fl 395.381 Mbit/s
sw_fr sw_c 450.938 Mbit/s
sw_fl zc_fl 418.456 Mbit/s
sw_fl infotainment 114.332 Mbit/s
sw_rr adas 1640.390 Mbit/s
sw_rr zc_rr 343.065 Mbit/s
sw_fl sw_c 249.928 Mbit/s
sw_c sw_rr 1632.342 Mbit/s
sw_fr zc_fr 390.200 Mbit/s
sw_c sw_fr 379.106 Mbit/s
sw_rr sw_c 200.810 Mbit/s
sw_c sw_rl 147.877 Mbit/s
sw_rl zc_rl 153.175 Mbit/s
sw_rl sw_c 318.233 Mbit/s
